# 3. Post-Submission Review

从提交结果 (`data/my_trades/`) 中加载 orderbook 快照、我方成交、PnL，绘制复盘图：

1. **主图**：Orderbook 散点 + 市场成交 + 我方买卖（共享横轴）
2. **仓位图**：逐笔累计仓位变化
3. **PnL 曲线**：总 PnL + 分产品 PnL
4. **交易质量分析**：成交偏离 fair 的分布、fill rate、每笔 edge 统计

In [138]:
import sys, json, io
sys.path.insert(0, '.')

from pathlib import Path
from utils.orderbook import compute_wall_mid
import polars as pl
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Step 1: 加载提交数据

从 `.log` 文件解析：`activitiesLog`（orderbook 快照）、`tradeHistory`（所有成交）、从 `.json` 解析 `graphLog`（总 PnL 曲线）。

In [139]:
# === 配置 ===
SUBMISSION_ID = "risk"
MY_TRADES_DIR = Path("data/my_trades")

# 加载 .log（含 tradeHistory）和 .json（含 graphLog / profit）
with open(MY_TRADES_DIR / f"{SUBMISSION_ID}.log") as f:
    log_data = json.loads(f.read())
with open(MY_TRADES_DIR / f"{SUBMISSION_ID}.json") as f:
    json_data = json.load(f)

print(f"Round: {json_data['round']}, Status: {json_data['status']}, Final Profit: {json_data['profit']}")
print(f"Final positions: {json_data['positions']}")

Round: 0, Status: FINISHED, Final Profit: 2556.0078125
Final positions: [{'symbol': 'XIRECS', 'quantity': 57580}, {'symbol': 'TOMATOES', 'quantity': 7}, {'symbol': 'EMERALDS', 'quantity': -9}]


## Step 2: 解析数据

In [140]:
# --- 1) Orderbook 快照（宽格式 + 长格式）---
ob_wide = pl.read_csv(io.StringIO(log_data["activitiesLog"]), separator=";")

# 宽格式 -> 长格式（每档一行），用于散点图
rows = []
for side in ["bid", "ask"]:
    for level in range(1, 4):
        pc, vc = f"{side}_price_{level}", f"{side}_volume_{level}"
        if pc in ob_wide.columns:
            rows.append(
                ob_wide.select([
                    "day", "timestamp", "product",
                    pl.col(pc).alias("price"),
                    pl.col(vc).alias("volume"),
                    pl.lit(side).alias("side"),
                    pl.lit(level).alias("level"),
                    "mid_price", "profit_and_loss",
                ])
            )
ob_long = pl.concat(rows).filter(pl.col("price").is_not_null()).sort(["timestamp", "product", "side", "level"])

# --- 2) 成交记录 ---
all_trades = pl.DataFrame(log_data["tradeHistory"]).rename({"symbol": "product"})
# 分类：我方 vs 市场
all_trades = all_trades.with_columns([
    pl.when(pl.col("buyer") == "SUBMISSION").then(pl.lit("my_buy"))
      .when(pl.col("seller") == "SUBMISSION").then(pl.lit("my_sell"))
      .otherwise(pl.lit("market"))
      .alias("trade_type"),
])

my_trades = all_trades.filter(pl.col("trade_type") != "market")
mkt_trades = all_trades.filter(pl.col("trade_type") == "market")

# --- 3) PnL 曲线（总额）---
pnl_total = pl.read_csv(io.StringIO(json_data["graphLog"]), separator=";")

# --- 4) Wall mid ---
products = ob_wide["product"].unique().to_list()
wall_mid_df = compute_wall_mid(ob_wide)

print(f"Orderbook snapshots: {ob_wide.height}, Products: {products}")
print(f"My trades: {my_trades.height} ({my_trades.filter(pl.col('trade_type')=='my_buy').height} buys, "
      f"{my_trades.filter(pl.col('trade_type')=='my_sell').height} sells)")
print(f"Market trades: {mkt_trades.height}")
print(f"PnL datapoints: {pnl_total.height}")
my_trades.head(5)

Orderbook snapshots: 4000, Products: ['TOMATOES', 'EMERALDS']
My trades: 138 (67 buys, 71 sells)
Market trades: 5
PnL datapoints: 500


timestamp,buyer,seller,product,currency,price,quantity,trade_type
i64,str,str,str,str,f64,i64,str
2900,"""SUBMISSION""","""""","""TOMATOES""","""XIRECS""",4998.0,3,"""my_buy"""
3300,"""SUBMISSION""","""""","""TOMATOES""","""XIRECS""",4997.0,3,"""my_buy"""
3900,"""""","""SUBMISSION""","""TOMATOES""","""XIRECS""",5002.0,8,"""my_sell"""
5900,"""""","""SUBMISSION""","""EMERALDS""","""XIRECS""",10007.0,8,"""my_sell"""
9100,"""SUBMISSION""","""""","""EMERALDS""","""XIRECS""",10000.0,7,"""my_buy"""


In [141]:
# --- 5) 计算逐笔仓位序列 ---
# 我方买入 = +quantity，我方卖出 = -quantity
position_rows = []
for product in products:
    pt = my_trades.filter(pl.col("product") == product).sort("timestamp")
    cum_pos = 0
    # 初始仓位 = 0
    position_rows.append({"timestamp": 0, "product": product, "position": 0})
    for row in pt.iter_rows(named=True):
        signed_qty = row["quantity"] if row["trade_type"] == "my_buy" else -row["quantity"]
        cum_pos += signed_qty
        position_rows.append({"timestamp": row["timestamp"], "product": product, "position": cum_pos})

position_df = pl.DataFrame(position_rows).sort(["product", "timestamp"])

# --- 6) 分产品 PnL（从 activitiesLog 的 profit_and_loss 列）---
pnl_by_product = (
    ob_wide.select(["timestamp", "product", "profit_and_loss"])
    .sort(["product", "timestamp"])
)

# --- 7) 每笔我方成交加上 fair / edge（后续 normalized 图 + Step 5 edge 分析都会用到）---
my_with_wm = my_trades.join(
    wall_mid_df.select(["timestamp", "product", "wall_mid"]),
    on=["timestamp", "product"],
    how="left",
).with_columns(
    pl.when(pl.col("product") == "EMERALDS")
      .then(pl.lit(10000.0))
      .otherwise(pl.col("wall_mid"))
      .alias("fair")
).with_columns(
    pl.when(pl.col("trade_type") == "my_buy")
      .then(pl.col("fair") - pl.col("price"))
      .otherwise(pl.col("price") - pl.col("fair"))
      .alias("edge"),
).with_columns(
    (pl.col("edge") * pl.col("quantity")).alias("edge_pnl"),
)

print("Position range per product:")
for product in products:
    pp = position_df.filter(pl.col("product") == product)
    print(f"  {product}: [{pp['position'].min()}, {pp['position'].max()}]")

print(f"\nFinal PnL per product:")
for product in products:
    pp = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    print(f"  {product}: {pp['profit_and_loss'].to_list()[-1]:.2f}")


Position range per product:
  TOMATOES: [-8, 17]
  EMERALDS: [-15, 19]

Final PnL per product:
  TOMATOES: 1506.01
  EMERALDS: 1050.00


## Step 3: 复盘主图

每个产品绘制三行子图（共享横轴）：
- **Row 1**：Orderbook 散点 + Wall Mid + 市场成交 + 我方买卖
- **Row 2**：仓位变化（阶梯线）
- **Row 3**：分产品 PnL 曲线

In [142]:
def plot_review(product: str) -> go.Figure:
    """三行子图：Orderbook+Trades / Position / PnL，共享横轴。"""

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.55, 0.2, 0.25],
        subplot_titles=[
            f"{product} — Orderbook + Trades",
            "Position",
            "PnL",
        ],
    )

    # ===== Row 1: Orderbook 散点 =====
    ob_p = ob_long.filter(pl.col("product") == product)
    max_vol = ob_p["volume"].max() or 1

    bids = ob_p.filter(pl.col("side") == "bid")
    asks = ob_p.filter(pl.col("side") == "ask")

    fig.add_trace(go.Scatter(
        x=bids["timestamp"].to_list(), y=bids["price"].to_list(),
        mode="markers",
        marker=dict(color="steelblue", size=[max(2, v / max_vol * 14) for v in bids["volume"].to_list()], opacity=0.4),
        name="Bids", legendgroup="ob",
        hovertemplate="t=%{x}<br>p=%{y}<extra>Bid</extra>",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=asks["timestamp"].to_list(), y=asks["price"].to_list(),
        mode="markers",
        marker=dict(color="salmon", size=[max(2, v / max_vol * 14) for v in asks["volume"].to_list()], opacity=0.4),
        name="Asks", legendgroup="ob",
        hovertemplate="t=%{x}<br>p=%{y}<extra>Ask</extra>",
    ), row=1, col=1)

    # Wall mid line
    wm = wall_mid_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=wm["timestamp"].to_list(), y=wm["wall_mid"].to_list(),
        mode="lines", line=dict(color="green", width=1.5),
        name="Wall Mid", legendgroup="wm",
    ), row=1, col=1)

    # 市场成交（黑色 x）
    mt = mkt_trades.filter(pl.col("product") == product)
    if mt.height > 0:
        fig.add_trace(go.Scatter(
            x=mt["timestamp"].to_list(), y=mt["price"].to_list(),
            mode="markers", marker=dict(color="black", symbol="x", size=7),
            name="Market Trades", legendgroup="mkt",
            customdata=mt["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>Mkt</extra>",
        ), row=1, col=1)

    # 我方买入（绿色三角上）
    my_buys = my_trades.filter((pl.col("product") == product) & (pl.col("trade_type") == "my_buy"))
    if my_buys.height > 0:
        fig.add_trace(go.Scatter(
            x=my_buys["timestamp"].to_list(), y=my_buys["price"].to_list(),
            mode="markers",
            marker=dict(color="limegreen", symbol="triangle-up", size=10, line=dict(width=1, color="darkgreen")),
            name="My Buy", legendgroup="my",
            customdata=my_buys["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>My Buy</extra>",
        ), row=1, col=1)

    # 我方卖出（红色三角下）
    my_sells = my_trades.filter((pl.col("product") == product) & (pl.col("trade_type") == "my_sell"))
    if my_sells.height > 0:
        fig.add_trace(go.Scatter(
            x=my_sells["timestamp"].to_list(), y=my_sells["price"].to_list(),
            mode="markers",
            marker=dict(color="red", symbol="triangle-down", size=10, line=dict(width=1, color="darkred")),
            name="My Sell", legendgroup="my",
            customdata=my_sells["quantity"].to_list(),
            hovertemplate="t=%{x}<br>p=%{y}<br>qty=%{customdata}<extra>My Sell</extra>",
        ), row=1, col=1)

    # ===== Row 2: 仓位 =====
    pos_p = position_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pos_p["timestamp"].to_list(), y=pos_p["position"].to_list(),
        mode="lines", line=dict(color="purple", width=1.5, shape="hv"),
        name="Position", legendgroup="pos",
        hovertemplate="t=%{x}<br>pos=%{y}<extra></extra>",
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=2, col=1)

    # ===== Row 3: PnL =====
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name=f"PnL ({product})", legendgroup="pnl",
        hovertemplate="t=%{x}<br>PnL=%{y:.1f}<extra></extra>",
    ), row=3, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=3, col=1)

    # Layout
    fig.update_layout(
        height=900, width=1200,
        legend=dict(orientation="h", y=1.02, x=0, font=dict(size=10)),
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="Timestamp", row=3, col=1)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Pos", row=2, col=1)
    fig.update_yaxes(title_text="PnL", row=3, col=1)

    return fig

In [143]:
fig = plot_review("EMERALDS")
fig.show()

In [144]:
fig = plot_review("TOMATOES")
fig.show()

### TOMATOES — Normalized View (去 wall_mid 漂移)

把所有价格减去成交时刻的 wall_mid，去除漂移后观察：
- 挂单 / 成交是否稳定落在 `± 7, ± 6, ± 1~2, 0` 等标准档位
- 我方成交相对 fair 的分布：buy 应该在 ≤ 0 的位置，sell 应该在 ≥ 0 的位置
- 负 edge 的成交会立刻暴露（buy 在 > 0 或 sell 在 < 0）

参考 [utils/viz.py](utils/viz.py) 中的 `plot_normalized_orderbook`，保留所有元素（bids / asks / market trades / my buys / my sells）+ 叠加仓位 / PnL 两行子图。

In [145]:
def plot_review_normalized(product: str) -> go.Figure:
    """和 plot_review 一样的三行布局，但 Row 1 使用 price - wall_mid（去漂移）。"""

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.55, 0.2, 0.25],
        subplot_titles=[
            f"{product} — Normalized Orderbook (price - wall_mid)",
            "Position",
            "PnL",
        ],
    )

    # ===== Row 1: 去漂移 Orderbook =====
    wm = wall_mid_df.filter(pl.col("product") == product).select(["timestamp", "product", "wall_mid"])

    # 挂单 - wall_mid
    ob_p = (
        ob_long.filter(pl.col("product") == product)
        .join(wm, on=["timestamp", "product"], how="left")
        .filter(pl.col("wall_mid").is_not_null())
        .with_columns((pl.col("price") - pl.col("wall_mid")).alias("norm_price"))
    )
    max_vol = ob_p["volume"].max() or 1

    bids = ob_p.filter(pl.col("side") == "bid")
    fig.add_trace(go.Scatter(
        x=bids["timestamp"].to_list(), y=bids["norm_price"].to_list(),
        mode="markers",
        marker=dict(color="steelblue", size=[max(2, v / max_vol * 14) for v in bids["volume"].to_list()], opacity=0.4),
        name="Bids", legendgroup="ob",
        customdata=bids["volume"].to_list(),
        hovertemplate="t=%{x}<br>Δ=%{y}<br>v=%{customdata}<extra>Bid</extra>",
    ), row=1, col=1)

    asks = ob_p.filter(pl.col("side") == "ask")
    fig.add_trace(go.Scatter(
        x=asks["timestamp"].to_list(), y=asks["norm_price"].to_list(),
        mode="markers",
        marker=dict(color="salmon", size=[max(2, v / max_vol * 14) for v in asks["volume"].to_list()], opacity=0.4),
        name="Asks", legendgroup="ob",
        customdata=asks["volume"].to_list(),
        hovertemplate="t=%{x}<br>Δ=%{y}<br>v=%{customdata}<extra>Ask</extra>",
    ), row=1, col=1)

    # Fair 基准线 (y=0)
    fig.add_hline(y=0, line_color="green", line_dash="dash", line_width=1, row=1, col=1,
                  annotation_text="Fair (wall_mid)")

    # 市场成交
    mt = (
        mkt_trades.filter(pl.col("product") == product)
        .join(wm, on=["timestamp", "product"], how="left")
        .filter(pl.col("wall_mid").is_not_null())
        .with_columns((pl.col("price") - pl.col("wall_mid")).alias("norm_price"))
    )
    if mt.height > 0:
        fig.add_trace(go.Scatter(
            x=mt["timestamp"].to_list(), y=mt["norm_price"].to_list(),
            mode="markers", marker=dict(color="black", symbol="x", size=7),
            name="Market Trades", legendgroup="mkt",
            customdata=mt["quantity"].to_list(),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata}<extra>Mkt</extra>",
        ), row=1, col=1)

    # 我方买入 / 卖出（用 my_with_wm 里已算好的 fair 作为参照，避免 wall_mid 找不到 timestamp）
    my_p = my_with_wm.filter(pl.col("product") == product).with_columns(
        (pl.col("price") - pl.col("fair")).alias("norm_price")
    )

    my_buys = my_p.filter(pl.col("trade_type") == "my_buy")
    if my_buys.height > 0:
        fig.add_trace(go.Scatter(
            x=my_buys["timestamp"].to_list(), y=my_buys["norm_price"].to_list(),
            mode="markers",
            marker=dict(color="limegreen", symbol="triangle-up", size=10, line=dict(width=1, color="darkgreen")),
            name="My Buy", legendgroup="my",
            customdata=list(zip(my_buys["quantity"].to_list(), my_buys["price"].to_list(), my_buys["edge"].to_list())),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata[0]}<br>px=%{customdata[1]}<br>edge=%{customdata[2]}<extra>My Buy</extra>",
        ), row=1, col=1)

    my_sells = my_p.filter(pl.col("trade_type") == "my_sell")
    if my_sells.height > 0:
        fig.add_trace(go.Scatter(
            x=my_sells["timestamp"].to_list(), y=my_sells["norm_price"].to_list(),
            mode="markers",
            marker=dict(color="red", symbol="triangle-down", size=10, line=dict(width=1, color="darkred")),
            name="My Sell", legendgroup="my",
            customdata=list(zip(my_sells["quantity"].to_list(), my_sells["price"].to_list(), my_sells["edge"].to_list())),
            hovertemplate="t=%{x}<br>Δ=%{y}<br>qty=%{customdata[0]}<br>px=%{customdata[1]}<br>edge=%{customdata[2]}<extra>My Sell</extra>",
        ), row=1, col=1)

    # 右侧副轴：wall_mid 趋势，用于观察价格漂移
    wm_sorted = wm.sort("timestamp")
    fig.add_trace(go.Scatter(
        x=wm_sorted["timestamp"].to_list(), y=wm_sorted["wall_mid"].to_list(),
        mode="lines", line=dict(color="green", width=1.2, dash="dot"),
        name="Wall Mid Trend", legendgroup="wm",
        yaxis="y4",
        hovertemplate="t=%{x}<br>wm=%{y}<extra></extra>",
    ))

    # ===== Row 2: 仓位 =====
    pos_p = position_df.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pos_p["timestamp"].to_list(), y=pos_p["position"].to_list(),
        mode="lines", line=dict(color="purple", width=1.5, shape="hv"),
        name="Position", legendgroup="pos",
        hovertemplate="t=%{x}<br>pos=%{y}<extra></extra>",
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=2, col=1)

    # ===== Row 3: PnL =====
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")
    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name=f"PnL ({product})", legendgroup="pnl",
        hovertemplate="t=%{x}<br>PnL=%{y:.1f}<extra></extra>",
    ), row=3, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=3, col=1)

    # 配置副轴 (wall mid trend)
    fig.update_layout(
        yaxis4=dict(
            overlaying="y", side="right", title="Wall Mid",
            showgrid=False, anchor="x",
        ),
        height=900, width=1200,
        legend=dict(orientation="h", y=1.02, x=0, font=dict(size=10)),
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="Timestamp", row=3, col=1)
    fig.update_yaxes(title_text="Δ from Fair (ticks)", row=1, col=1)
    fig.update_yaxes(title_text="Pos", row=2, col=1)
    fig.update_yaxes(title_text="PnL", row=3, col=1)

    return fig


fig = plot_review_normalized("TOMATOES")
fig.show()

## Step 4: 总 PnL 曲线

graphLog 提供的全产品汇总 PnL 曲线，叠加分产品 PnL 对比。

In [146]:
# my_with_wm 已在 Step 2 里算好（含 fair / edge / edge_pnl），这里直接做统计
for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    print(f"\n{'='*40} {product} {'='*40}")
    print(f"  Total trades: {pt.height}")
    print(f"  Positive edge trades: {pt.filter(pl.col('edge') > 0).height}")
    print(f"  Zero edge trades: {pt.filter(pl.col('edge') == 0).height}")
    print(f"  Negative edge trades: {pt.filter(pl.col('edge') < 0).height}")
    print(f"  Avg edge: {pt['edge'].mean():.2f}")
    print(f"  Total edge PnL (sum of edge*qty): {pt['edge_pnl'].sum():.2f}")
    print(f"  Edge distribution:")
    edge_dist = pt.group_by("edge").agg([
        pl.len().alias("count"),
        pl.col("quantity").sum().alias("total_qty"),
    ]).sort("edge")
    print(edge_dist)



======================================== TOMATOES ========================================
  Total trades: 87
  Positive edge trades: 79
  Zero edge trades: 8
  Negative edge trades: 0
  Avg edge: 4.61
  Total edge PnL (sum of edge*qty): 1494.50
  Edge distribution:
shape: (7, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │
│ ---  ┆ ---   ┆ ---       │
│ f64  ┆ u32   ┆ i64       │
╞══════╪═══════╪═══════════╡
│ 0.0  ┆ 8     ┆ 70        │
│ 1.0  ┆ 7     ┆ 57        │
│ 1.5  ┆ 1     ┆ 12        │
│ 2.0  ┆ 6     ┆ 58        │
│ 5.0  ┆ 7     ┆ 22        │
│ 5.5  ┆ 4     ┆ 13        │
│ 6.0  ┆ 54    ┆ 187       │
└──────┴───────┴───────────┘

======================================== EMERALDS ========================================
  Total trades: 51
  Positive edge trades: 29
  Zero edge trades: 22
  Negative edge trades: 0
  Avg edge: 3.98
  Total edge PnL (sum of edge*qty): 1050.00
  Edge distribution:
shape: (2, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │


## Step 5: 交易质量分析

### 5a — 每笔成交的 Edge（trade_price - fair）

正 edge = 赚钱方向成交。对 EMERALDS fair=10000 固定，对 TOMATOES 用成交时刻的 wall_mid。

In [147]:
# 给每笔我方成交加上 wall_mid，计算 edge
my_with_wm = my_trades.join(
    wall_mid_df.select(["timestamp", "product", "wall_mid"]),
    on=["timestamp", "product"],
    how="left",
)

# 对于找不到精确匹配的 timestamp，用 EMERALDS 的固定 fair 补齐
my_with_wm = my_with_wm.with_columns(
    pl.when(pl.col("product") == "EMERALDS")
      .then(pl.lit(10000.0))
      .otherwise(pl.col("wall_mid"))
      .alias("fair")
)

# edge = 买入 → fair - price（买越低 edge 越正）；卖出 → price - fair
my_with_wm = my_with_wm.with_columns(
    pl.when(pl.col("trade_type") == "my_buy")
      .then(pl.col("fair") - pl.col("price"))
      .otherwise(pl.col("price") - pl.col("fair"))
      .alias("edge"),
).with_columns(
    (pl.col("edge") * pl.col("quantity")).alias("edge_pnl"),  # edge × qty
)

for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    print(f"\n{'='*40} {product} {'='*40}")
    print(f"  Total trades: {pt.height}")
    print(f"  Positive edge trades: {pt.filter(pl.col('edge') > 0).height}")
    print(f"  Zero edge trades: {pt.filter(pl.col('edge') == 0).height}")
    print(f"  Negative edge trades: {pt.filter(pl.col('edge') < 0).height}")
    print(f"  Avg edge: {pt['edge'].mean():.2f}")
    print(f"  Total edge PnL (sum of edge*qty): {pt['edge_pnl'].sum():.2f}")
    print(f"  Edge distribution:")
    edge_dist = pt.group_by("edge").agg([
        pl.len().alias("count"),
        pl.col("quantity").sum().alias("total_qty"),
    ]).sort("edge")
    print(edge_dist)


======================================== TOMATOES ========================================
  Total trades: 87
  Positive edge trades: 79
  Zero edge trades: 8
  Negative edge trades: 0
  Avg edge: 4.61
  Total edge PnL (sum of edge*qty): 1494.50
  Edge distribution:
shape: (7, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │
│ ---  ┆ ---   ┆ ---       │
│ f64  ┆ u32   ┆ i64       │
╞══════╪═══════╪═══════════╡
│ 0.0  ┆ 8     ┆ 70        │
│ 1.0  ┆ 7     ┆ 57        │
│ 1.5  ┆ 1     ┆ 12        │
│ 2.0  ┆ 6     ┆ 58        │
│ 5.0  ┆ 7     ┆ 22        │
│ 5.5  ┆ 4     ┆ 13        │
│ 6.0  ┆ 54    ┆ 187       │
└──────┴───────┴───────────┘

======================================== EMERALDS ========================================
  Total trades: 51
  Positive edge trades: 29
  Zero edge trades: 22
  Negative edge trades: 0
  Avg edge: 3.98
  Total edge PnL (sum of edge*qty): 1050.00
  Edge distribution:
shape: (2, 3)
┌──────┬───────┬───────────┐
│ edge ┆ count ┆ total_qty │


### 5b — Edge 散点时序图

每笔成交的 edge 随时间变化，观察是否有时段 edge 恶化。

In [148]:
fig = make_subplots(rows=len(products), cols=1, shared_xaxes=True,
                    subplot_titles=[f"{p} — Edge per Trade" for p in products],
                    vertical_spacing=0.08)

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product).sort("timestamp")
    if pt.height == 0:
        continue

    # 颜色：正 edge 绿，零 edge 灰，负 edge 红
    colors = ["green" if e > 0 else ("gray" if e == 0 else "red") for e in pt["edge"].to_list()]

    fig.add_trace(go.Scatter(
        x=pt["timestamp"].to_list(), y=pt["edge"].to_list(),
        mode="markers",
        marker=dict(color=colors, size=8, line=dict(width=0.5, color="black")),
        customdata=list(zip(pt["quantity"].to_list(), pt["trade_type"].to_list(), pt["price"].to_list())),
        hovertemplate="t=%{x}<br>edge=%{y}<br>qty=%{customdata[0]}<br>%{customdata[1]} @ %{customdata[2]}<extra></extra>",
        name=product, showlegend=False,
    ), row=i, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=i, col=1)
    fig.update_yaxes(title_text="Edge", row=i, col=1)

fig.update_xaxes(title_text="Timestamp", row=len(products), col=1)
fig.update_layout(height=350 * len(products), width=1200)
fig.show()

### 5c — 成交档位分布（normalized level histogram）

我方成交落在哪些 `price - fair` 档位，和预期策略设计对比。

In [149]:
fig = make_subplots(rows=1, cols=len(products),
                    subplot_titles=[f"{p} — Fill Level Distribution" for p in products])

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product)
    if pt.height == 0:
        continue

    # norm_level = price - fair; 买入为负 edge，卖出为正 edge
    norm = (pt["price"] - pt["fair"]).to_list()

    # 分 buy/sell 颜色
    buys_norm = pt.filter(pl.col("trade_type") == "my_buy")
    sells_norm = pt.filter(pl.col("trade_type") == "my_sell")

    if buys_norm.height > 0:
        fig.add_trace(go.Histogram(
            x=(buys_norm["price"] - buys_norm["fair"]).to_list(),
            marker_color="limegreen", opacity=0.7, name="Buy", legendgroup="buy",
            showlegend=(i == 1),
        ), row=1, col=i)
    if sells_norm.height > 0:
        fig.add_trace(go.Histogram(
            x=(sells_norm["price"] - sells_norm["fair"]).to_list(),
            marker_color="red", opacity=0.7, name="Sell", legendgroup="sell",
            showlegend=(i == 1),
        ), row=1, col=i)

    fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=1, row=1, col=i)
    fig.update_xaxes(title_text="Price - Fair", row=1, col=i)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(height=400, width=1200, barmode="overlay",
                  legend=dict(orientation="h", y=1.08, x=0))
fig.show()

## Step 6: 成交间隔 & 活跃度分析

观察我方成交的时间分布：是否均匀地参与了全天成交机会？空闲时段是否是因为没有机会还是策略遗漏？

In [150]:
# 成交间隔直方图（每产品）
fig = make_subplots(rows=1, cols=len(products),
                    subplot_titles=[f"{p} — Trade Interval (ms)" for p in products])

for i, product in enumerate(products, 1):
    pt = my_trades.filter(pl.col("product") == product).sort("timestamp")
    if pt.height < 2:
        continue
    intervals = pt["timestamp"].diff().drop_nulls().to_list()

    fig.add_trace(go.Histogram(
        x=intervals, nbinsx=40, marker_color="steelblue",
        name=product, showlegend=False,
    ), row=1, col=i)
    fig.update_xaxes(title_text="Interval (ms)", row=1, col=i)

    mean_iv = np.mean(intervals)
    median_iv = np.median(intervals)
    print(f"{product}: {len(intervals)} intervals, mean={mean_iv:.0f}ms, median={median_iv:.0f}ms, "
          f"max={max(intervals)}ms, max_gap_at_ts={pt['timestamp'].to_list()[np.argmax(intervals)+1]}")

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(height=350, width=1200)
fig.show()

TOMATOES: 86 intervals, mean=2274ms, median=1750ms, max=9900ms, max_gap_at_ts=153600
EMERALDS: 50 intervals, mean=3786ms, median=2500ms, max=24300ms, max_gap_at_ts=142300


## Step 7: PnL 归因 — 做市 vs 头寸损益

将 PnL 拆分为：
- **Realized edge PnL**：每笔成交时的 edge × qty 累计（确定性收益）
- **Mark-to-market PnL**：持仓的未实现盈亏（随价格漂移波动）

两者之和应近似等于总 PnL。

In [151]:
fig = make_subplots(rows=len(products), cols=1, shared_xaxes=True,
                    subplot_titles=[f"{p} — PnL Attribution" for p in products],
                    vertical_spacing=0.08)

for i, product in enumerate(products, 1):
    pt = my_with_wm.filter(pl.col("product") == product).sort("timestamp")
    if pt.height == 0:
        continue

    # 累计 edge PnL
    ts_list = pt["timestamp"].to_list()
    cum_edge = np.cumsum(pt["edge_pnl"].to_list())

    # 总 PnL（从 activitiesLog）
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")

    # Mark-to-market 差值 = total_pnl - cum_edge（在成交时刻内插）
    fig.add_trace(go.Scatter(
        x=ts_list, y=cum_edge.tolist(),
        mode="lines+markers", line=dict(color="green", width=1.5),
        marker=dict(size=4),
        name="Cum Edge PnL", legendgroup="edge", showlegend=(i == 1),
    ), row=i, col=1)

    fig.add_trace(go.Scatter(
        x=pnl_p["timestamp"].to_list(), y=pnl_p["profit_and_loss"].to_list(),
        mode="lines", line=dict(color="orange", width=1.5),
        name="Total PnL", legendgroup="total", showlegend=(i == 1),
    ), row=i, col=1)

    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=0.5, row=i, col=1)
    fig.update_yaxes(title_text="PnL", row=i, col=1)

    final_edge = cum_edge[-1] if len(cum_edge) else 0
    final_total = pnl_p["profit_and_loss"].to_list()[-1] if pnl_p.height else 0
    print(f"{product}: cum_edge={final_edge:.2f}, total_pnl={final_total:.2f}, "
          f"mtm_component={final_total - final_edge:.2f}")

fig.update_xaxes(title_text="Timestamp", row=len(products), col=1)
fig.update_layout(height=400 * len(products), width=1200,
                  legend=dict(orientation="h", y=1.03, x=0))
fig.show()

TOMATOES: cum_edge=1494.50, total_pnl=1506.01, mtm_component=11.51
EMERALDS: cum_edge=1050.00, total_pnl=1050.00, mtm_component=0.00


## Step 8: 汇总统计表

一目了然的关键指标。

In [152]:
summary_rows = []
for product in products:
    pt = my_with_wm.filter(pl.col("product") == product)
    pos_p = position_df.filter(pl.col("product") == product)
    pnl_p = pnl_by_product.filter(pl.col("product") == product).sort("timestamp")

    n_trades = pt.height
    total_vol = pt["quantity"].sum() if n_trades > 0 else 0
    n_pos_edge = pt.filter(pl.col("edge") > 0).height
    n_zero_edge = pt.filter(pl.col("edge") == 0).height
    n_neg_edge = pt.filter(pl.col("edge") < 0).height
    avg_edge = pt["edge"].mean() if n_trades > 0 else 0
    cum_edge_pnl = pt["edge_pnl"].sum() if n_trades > 0 else 0
    final_pnl = pnl_p["profit_and_loss"].to_list()[-1] if pnl_p.height else 0
    max_pos = pos_p["position"].max()
    min_pos = pos_p["position"].min()
    final_pos = pos_p["position"].to_list()[-1]

    # 成交频率（trades per 1000 ts）
    if n_trades >= 2:
        ts_range = pt["timestamp"].max() - pt["timestamp"].min()
        freq = n_trades / (ts_range / 1000) if ts_range > 0 else 0
    else:
        freq = 0

    summary_rows.append({
        "product": product,
        "trades": n_trades,
        "total_vol": total_vol,
        "pos_edge": n_pos_edge,
        "zero_edge": n_zero_edge,
        "neg_edge": n_neg_edge,
        "avg_edge": round(avg_edge, 2),
        "cum_edge_pnl": round(cum_edge_pnl, 2),
        "final_pnl": round(final_pnl, 2),
        "mtm_component": round(final_pnl - cum_edge_pnl, 2),
        "max_pos": max_pos,
        "min_pos": min_pos,
        "final_pos": final_pos,
        "trades_per_1k_ts": round(freq, 2),
    })

summary_df = pl.DataFrame(summary_rows)
print(summary_df)

shape: (2, 14)
┌──────────┬────────┬───────────┬──────────┬───┬─────────┬─────────┬───────────┬──────────────────┐
│ product  ┆ trades ┆ total_vol ┆ pos_edge ┆ … ┆ max_pos ┆ min_pos ┆ final_pos ┆ trades_per_1k_ts │
│ ---      ┆ ---    ┆ ---       ┆ ---      ┆   ┆ ---     ┆ ---     ┆ ---       ┆ ---              │
│ str      ┆ i64    ┆ i64       ┆ i64      ┆   ┆ i64     ┆ i64     ┆ i64       ┆ f64              │
╞══════════╪════════╪═══════════╪══════════╪═══╪═════════╪═════════╪═══════════╪══════════════════╡
│ TOMATOES ┆ 87     ┆ 419       ┆ 79       ┆ … ┆ 17      ┆ -8      ┆ 7         ┆ 0.44             │
│ EMERALDS ┆ 51     ┆ 257       ┆ 29       ┆ … ┆ 19      ┆ -15     ┆ -9        ┆ 0.27             │
└──────────┴────────┴───────────┴──────────┴───┴─────────┴─────────┴───────────┴──────────────────┘
